## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report


![Architecture](data/architecture.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [ ]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [3]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

#### ✅ Answer

#### State Interrelationships
As previously stated, the state structure follows a Hierarchical Flow: Agent → Supervisor → Researcher (multiple instances)

Data Flow:
- **Down**: In write_clarification_brief node we update **AgentState** fields with **research_brief** and **supervisor_messages** (containing the supervisor system prompt and the detailed brief) and **delegate to supervisor**. In supervisor node we read from updated state (supervisor_messages), generate a response from reading the brief, update the **supervisor_messages** field from **SupervisorState** with the response. This response will contain the detailed plan as result of using think_tool and the subsequent tool calls needed as a result of ConductResearch or a signal that the research is complete. We  proceed to supervisor_tools node. If any, the tool calls are extracted, if there is a tool call for conducting research (which is a structured tool defined with pydantic with a field called research_topic) the research subgraph is invoked asynchronously. From here we invoke dynamically parallel researches that invoke a response on messages (system context with researcher prompt + research messages), update the field **research_messages** from **ResearchState** (and tool_call iterations) and go to the node **researcher_tools**. In **research_tools** node the **ResearchState** field **research_messages** is updated with the tool outputs. Next goes the **compressed_research** node that takes the **researcher_messages** from **ResearcherState** plus the compression_prompt synthesizes the key findings. We return the **compressed_research** and the **raw notes**.

- **Up**: Back to supervisor subgraph, we update **supervisor_messages** with the compressed_research and the field **raw_notes** (aggregation). If the research is complete (we exceeded the allow number of iteration, tokens or there are no more tool calls), we update SupervisorState's research brief field and notes from tool calls and go up to main graph's node  final_report_generation. Here we populate the AgentState's final report field, add this to **messages** and clear the state.

#### Why Not a Single State?
1. Parallel Execution: Multiple researchers must run concurrently - single state would create race conditions (multiple processes modify the same data)
2. Separation of Concerns:
- Agent: User interaction, final reports
- Supervisor: Research strategy, task coordination
- Researcher: Tool execution, information gathering
3. Different Lifecycles: Each has unique exit conditions and iteration limits
4. Error Isolation: Failures in one level don't crash others
5. Resource Management: Granular control over concurrent research units and token limits
6. State Reducers: Different data management needs (add vs override)
A single state would be a monolithic mess that prevents parallelization, creates race conditions, and makes the system unmaintainable. We want to avoid 'context rot'.




## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [4]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:

### ✅ Answer

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

**Advantages**: Importing is better for production systems where code organization, reusability, and maintainability matter most. It makes the process of remove or add structure to the Agentic application easier.

**Disadvantages**: Reduced transparency (logic is hidden), making it harder to follow or tweak the flow; it also causes debugging blindness since errors point into library code, and the notebook isn’t fully self-contained (extra setup/versioning needed)

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [5]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [6]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [7]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [37]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [41]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [11]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [12]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [13]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [14]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [48]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [19]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." I understand you want me to provide insights on:

1. Main findings about how people are using AI (ChatGPT specifically)
2. Most common use cases identified in the study
3. Trends and patterns that emerge from the data

The document provides comprehensive data from ChatGPT usage between May 2024 and June 2025, including growth statistics, user demographics, work vs. non-work usage patterns, and detailed conversation classifications. I will now begin analyzing this research to provide you with the requested insights.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working paper "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl Yan Shan, and Kevin Wadman. Specifically, I wan


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# How People Use ChatGPT: Comprehensive Analysis of NBER Working Paper 34255

## Overview and Study Significance

The NBER working paper "How People Use ChatGPT" (Working Paper No. 34255) represents the most comprehensive analysis of consumer AI usage ever released. Published in September 2025 by Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl Yan Shan, and Kevin Wadman, this groundbreaking study documents ChatGPT's unprecedented global adoption and usage patterns from its November 2022 launch through July 2025 [1][2][3][4].

Drawing on 1.5 million conversations from consumer plans, the research reveals a technology that now serves roughly 700 million weekly users who send more than 2.5 billion messages per day [8]. The study was approved by Harvard IRB and conducted using strict privacy-preserving methodologies, ensuring no researcher ever saw actual message content or personal information [1][4].

## Main Findings: Adoption, Demographics, and Work vs. Non-Work Usage

### Unprecedented Global Adoption Statistics

By July 2025, ChatGPT achieved remarkable penetration, reaching approximately 10% of the world's adult population with over 700 million weekly active users [1][2][3][4][6][8]. The scale of usage is staggering: 18 billion messages were being sent each week by July 2025, translating to more than 2.6 billion messages per day, or over 30,000 messages per second [1][4].

The growth trajectory demonstrates no historical precedent for technological diffusion. ChatGPT reached one million registered users by December 5, 2022, just five days after its November 30 launch [1][4]. It achieved 100 million weekly active users in early November 2023, less than one year after release, and the user base has been doubling every 7-8 months since then [1]. Total message volume increased by 5.8x in the last year alone, with ChatGPT reaching the 1 billion message milestone in December 2024 [1].

### Demographic Evolution and Global Patterns

The demographic composition of ChatGPT users has undergone significant transformation. Early adopters were disproportionately male, with over 80% having typically masculine names initially. However, the gender gap narrowed dramatically over time, reaching gender parity by July 2025 when 52% of users had typically feminine names [1][2][3][4]. This represents a substantial shift from 37% feminine names in January 2024 to 52% by July 2025 [2].

Age distribution reveals that nearly half of all messages sent by adults come from users under 26, though age gaps have narrowed somewhat in recent months [4][6]. The global expansion pattern shows particularly rapid adoption in low- and middle-income countries, with growth rates 4x higher than in high-income countries by May 2025 [2]. Remarkably, countries at the 50th versus 90th percentile of GDP per capita now show similar usage rates despite vastly different income levels [1].

### The Shift from Work to Non-Work Usage

One of the study's most significant findings is the dramatic shift in usage patterns from work-related to non-work applications. In June 2024, work-related messages comprised 47% of total usage, but this declined to just 27% by June 2025, while non-work messages grew from 53% to 73% of all usage [2][3][4][6]. This represents steady growth in work-related messages but even faster growth in non-work applications.

Importantly, this decrease in work-related message share is primarily due to changing usage patterns within existing user cohorts rather than compositional changes in new users [4]. Work usage remains more common among educated users in highly-paid professional occupations, but the overall trend indicates ChatGPT's expanding role in personal, non-professional contexts [2][3][4].

### Usage Intensity and User Engagement Over Time

A key finding regarding user engagement is that usage intensity increases substantially over time. Early adopters from Q1 2023 were sending 40% more messages per day in July 2025 than they did two years earlier, suggesting ChatGPT has become better and more integrated into users' daily lives [1]. All user cohorts showed flat usage through 2024 followed by substantial increases beginning in early 2025, indicating both product improvements and user adaptation [1].

## Most Common Use Cases: The Three Dominant Categories

### The 80% Rule: Three Primary Usage Categories

The study's conversation classifier reveals that nearly 80% of all ChatGPT usage falls into three broad categories: Practical Guidance (29%), Writing (24%), and Seeking Information (24%) [2][3][4][6]. These categories have remained relatively stable over time, though their relative proportions have shifted.

### Practical Guidance: The Leading Use Case

**Practical Guidance** emerges as the most common use case at 29% of all usage, maintaining consistent prevalence over time [4]. This category encompasses:
- Tutoring and teaching activities (10.2% of all messages, representing 36% of Practical Guidance messages)
- How-to advice across various topics
- Creative ideation and brainstorming
- General advisory support

The tutoring and teaching component alone accounts for about 10% of all messages, highlighting education as a key application area for ChatGPT [4]. This suggests ChatGPT serves as a significant educational resource beyond formal learning environments.

### Writing: The Workplace Dominant

**Writing** accounts for 24% of overall usage but dominates work-related tasks, representing 40% of work-related messages in June 2025 [2][3][4]. This category has experienced some decline from 36% of all usage in July 2024 to 24% a year later [4]. Writing encompasses:
- Automated production of emails, documents, and communications
- Editing and critiquing existing text
- Summarizing and translating content
- Text modification and refinement

A crucial insight is that about two-thirds of all Writing messages involve modifying user-provided text (editing, critiquing, translating, summarizing) rather than creating entirely new content from scratch [4][9]. This pattern suggests users often seek assistance with refining and improving existing work rather than wholesale content generation.

### Seeking Information: The Growing Category

**Seeking Information** has shown remarkable growth, expanding from 14% to 24% of all usage between July 2024 and July 2025 [4]. This category includes:
- Searching for facts and current events
- Product research and comparisons
- Recipe and instructional lookups
- People and company information

The research notes that Seeking Information appears to be a very close substitute for traditional web search, but with the advantage of receiving customized, conversational responses rather than lists of links [4][9].

### Other Notable Categories

Beyond the dominant three, several other categories provide insight into ChatGPT usage patterns:
- **Technical Help** declined significantly from 12% to 5% of all usage and from 18% to 10% of work-related messages between July 2024 and July 2025 [4]
- **Multimedia** grew from 2% to just over 7%, with a large spike in April 2025 following new image-generation capabilities [4]
- **Computer Programming** accounts for only 4.2% of consumer conversations, along with Mathematical Calculations (3%) and Data Analysis (0.4%) [4][6][8]
- **Social and Personal** usage remains minimal, with only 2.4% of messages related to Relationships and Personal Reflection (1.9%) or Games and Role Play (0.4%) [4]

## Trends, Patterns, and Economic Implications

### User Intent Classification: Asking, Doing, Expressing

The researchers developed a novel taxonomy classifying messages by user intent using three categories: Asking, Doing, and Expressing [4]. The current distribution shows approximately 49% Asking, 40% Doing, and 11% Expressing, with Asking growing faster than Doing [4][9].

**Asking** involves seeking information or clarification to inform decisions, corresponding to problem-solving models of knowledge work [4][9]. **Doing** focuses on producing outputs or performing specific tasks, aligning with classic task-based work models [4][9]. **Expressing** involves sharing views or feelings without seeking information or action [4][9].

The evolution of these categories reveals important trends. In July 2024, usage was evenly split between Asking and Doing with under 8% Expressing. By June 2025, the split had shifted to 51.6% Asking, 34.6% Doing, and 13.8% Expressing [4]. Notably, Asking messages receive higher quality ratings than Doing messages from both automated classifiers and user feedback [6][9].

### Work-Related Intent Patterns

Work usage shows distinct patterns, with about 56% of work-related messages classified as Doing, and nearly three-quarters of those involving Writing tasks [4][6]. This means approximately 35% of all work-related queries are Doing messages related to Writing [4]. The prevalence of decision-support activities (Asking) in work contexts highlights ChatGPT's role beyond simple task completion.

### Economic Value and Consumer Surplus

The study provides compelling evidence of ChatGPT's economic value, particularly as a decision-support tool [6]. The research concludes that ChatGPT provides economic value primarily through decision support, which is especially important in knowledge-intensive jobs [2][3][4]. This finding aligns with research by Collis and Brynjolfsson (2025), who estimate a consumer surplus of at least $97 billion in 2024 alone in the US, based on users requiring roughly $98 to give up generative AI for a month [4][8].

### O*NET Work Activity Analysis

Analysis using O*NET work activities reveals that about 81% of work-related messages associate with two broad activities: (1) obtaining, documenting, and interpreting information; and (2) making decisions, giving advice, solving problems, and thinking creatively [4]. Remarkably, these work activities remain highly similar across very different occupations, from management and business to STEM to administrative and sales roles [4].

### Quality and User Satisfaction

User satisfaction metrics indicate strong positive reception, with positive interactions outnumbering negative ones by 4:1 [6]. The quality ratings consistently favor Asking over Doing messages, suggesting users derive particular value from ChatGPT's advisory capabilities rather than purely task-completion functions [9].

### Comparison with Traditional Search Engines

The study emphasizes ChatGPT's differentiation from traditional search engines through its ability to produce writing, software code, spreadsheets, and other digital products [4]. Even for traditional applications like Seeking Information and Practical Guidance, ChatGPT provides more flexibility through customized responses such as tailored workout plans, new product ideas, and personalized recommendations that represent newly generated or modified content [4].

## Methodology: Privacy-Preserving Automated Classification

### Automated Classification Pipeline

The study employed sophisticated privacy-preserving methodologies to analyze usage patterns without compromising user privacy. All message content analysis was performed via automated LLM-based classifiers run on de-identified and PII-scrubbed message data [4]. The classification system uses five different LLM-based classifiers, with messages categorized using "gpt-5-mini" model (except Interaction Quality, which uses "gpt-5") [4].

Each classification considers not just the randomly-selected user message but also the prior 10 messages in the conversation for context, with messages truncated to a maximum of 5,000 characters to maintain classification quality [4]. The taxonomies are defined through prompts passed to the LLM, enabling message classification without human review of content [4].

### Secure Data Clean Room Analysis

For employment and education analysis, the researchers used a secure data clean room protocol involving approximately 130,000 Free, Plus, and Pro users [4]. This approach allowed analysis of aggregated employment categories while maintaining strict privacy protections. The data clean room permitted only pre-approved aggregate computations across independently held datasets, with neither party able to view the other's underlying records [4].

The analysis enforced strict aggregation limits, approving only code returning cells with a minimum threshold of 100 users [4]. This methodology enabled insights about usage differences across demographic groups while protecting individual user privacy.

### Privacy Safeguards and Validation

The study implemented comprehensive privacy safeguards including automated PII removal, secure data analysis environments, and multiple approval cycles with public logging of all code [1]. For validation, researchers compared model classification decisions against human-judged classifications using the publicly available WildChat dataset, classifying a sample of 100,000 public messages for transparency [4].

### Study Limitations and Scope

The analysis focuses exclusively on consumer plans (Free, Plus, Pro), excluding business-oriented plans (Business, Enterprise, Education) [4]. Additional exclusions include users who opted out of message sharing for training, self-reported users under 18, deleted conversations, deactivated accounts, and logged-out users [4]. The researchers acknowledge that because data was available for only a subset of users, results may not represent the full user pool [4].

### Sources

[1] Forked Lightning Blog - How People Use ChatGPT: https://forklightning.substack.com/p/how-people-use-chatgpt

[2] OpenAI Blog - How people are using ChatGPT: https://openai.com/index/how-people-are-using-chatgpt/

[3] Korea Institute of Finance - How People Use ChatGPT: https://www.kif.re.kr/kif4/publication/redirect?mid=13&cno=353312

[4] NBER Working Paper W34255 - How People Use ChatGPT (PDF): https://www.nber.org/system/files/working_papers/w34255/w34255.pdf

[5] NBER Working Papers - How People Use ChatGPT: https://www.nber.org/papers/w34255

[6] TechManiacs - How People Really Use ChatGPT: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/

[7] LinkedIn Post by Dergal - NBER Study: https://www.linkedin.com/posts/dergal_new-nber-study-reveals-how-the-world-is-using-activity-7373740661918932992-e6RI

[8] BinaryVerse AI - What Is ChatGPT Used For: https://binaryverseai.com/what-is-chatgpt-used-for/

[9] Towards AI - How People Actually Use ChatGPT: https://pub.towardsai.net/how-people-actually-use-chatgpt-2790df683c00


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

#### ✅ Answer

To answer the activity 1, the main changes where:

- research model and summarization model: xai:grok-4-fast-reasoning-latest
- final report model: openai:gpt-5
- compression model: anthropic:claude-sonnet-4-20250514
- allow_clarification": False. Nonetheless, several times, even though this parameter was activated the model asked for clarification and stopped the flow as it
was not provided.
- The following parameters made the research deeper but increased test-time compute:
```python

"max_concurrent_research_units": 2,  # 2 parallel researchers (positive and negative sentiment)
        "max_researcher_iterations": 3,      # Supervisor can delegate up to 3 times: more comprehensive coverage 
        "max_react_tool_calls": 5,           # Each researcher can make up to 5 tool calls
```
- An image was given as context with the following modification in transform_messages_into_research_topic_prompt: 
```python
"""
6. Image Context
- If an image is provided as context, incorporate the visual elements into the research question.
- Consider how the visual context might influence the research direction.
- Include visual analysis as part of the research scope.
""
```

In [54]:

# os.environ["XAI_API_KEY"] = getpass.getpass("Enter your xAI API key: ")
# os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Use case: Sentiment Analysis of Taylor Swift:
import base64
from openai import OpenAI

client = OpenAI()

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# Path to your image
image_path = "data/taylor_swift.png"

# Getting the Base64 string
base64_image = encode_image(image_path)


response = client.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": "what's in this image?" },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                    "detail": "high"
                    
                },
            ],
        }
    ],
)

image_content=response.output_text


graph_2 = deep_researcher

config_2 = {
    "configurable": {
        # Model configuration - using Grok for everything
        "research_model": "xai:grok-4-fast-reasoning-latest",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "openai:gpt-5",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "xai:grok-4-fast-reasoning-latest",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 2,  # 2 parallel researchers (positive and negative sentiment)
        "max_researcher_iterations": 3,      # Supervisor can delegate up to 3 times: more comprehensive coverage 
        "max_react_tool_calls": 5,           # Each researcher can make up to 5 tool calls
        
        # Search configuration
        "search_api": "anthropic",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

# Create our research request with PDF context
research_request = f"""
Analyze the sentiment about Taylor Swift's latest album, the title and style is provided in the following image:
<Image>
{image_content}
</Image>

*IMPORTANT: if an image, pdf, document is provided as context. DO NOT ASK FOR CLARIFICATION. PROCEED DIRECTLY TO RESEARCH*

Use the image as visual context to understand:
- The album's visual themes and aesthetic
- The mood and tone conveyed by the artwork
- Any visual elements that might influence public sentiment

Then research the sentiment about this album by analyzing:
- Social media reactions and tweets.
- Fan discussions and reviews
- Media coverage and opinions
- Overall public sentiment (positive, negative, neutral)
- Key themes and topics mentioned
- Any controversies or debates

Focus on how the visual elements from the image might relate to the public sentiment.
"""



In [52]:
# Verify the model has extracted the correct information
image_content

'- An orange, glittery vinyl record partly in its sleeve.\n- The sleeve shows a woman in a beaded showgirl costume reclining in water, wearing bracelets.\n- Text on the cover/label reads: “THE LIFE OF A SHOWGIRL.”'

In [55]:
# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph_2.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config_2,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

Thank you for providing the details on Taylor Swift's album "THE LIFE OF A SHOWGIRL," including the visual description from the image. I have sufficient information to proceed, understanding the task involves analyzing the album's visual themes (e.g., glamorous showgirl aesthetic, glittery orange elements, and watery reclining pose), mood/tone, and their relation to public sentiment. I will now begin researching social media reactions, fan discussions, reviews, media coverage, overall sentiment, key themes, and any controversies.

Node: write_research_brief

Research Brief Generated:
As of October 13, 2025, I want to analyze the public sentiment toward Taylor Swift's latest album titled "THE LIFE OF A SHOWGIRL," incorporating the visual context from its album artwork: a vinyl record with a translucent, glittery orange pressing; the jacket featuring a glamorous woman reclining in water, dressed in a beaded showgirl-style outfit an

# Public Sentiment Analysis: Taylor Swift’s “THE LIFE OF A SHOWGIRL” (as of October 13, 2025)

## Executive Summary
The album artwork for “THE LIFE OF A SHOWGIRL” presents a glamorous, high-sheen showgirl aesthetic anchored by a translucent glittery orange vinyl, a water-reclining pose, and glittered title typography. The visuals communicate luxury, performance, spectacle, and dreamlike introspection—signals that typically prime audiences to expect a concept-forward era centered on persona, stagecraft, and the tension between public performance and private vulnerability. 

Public sentiment tied solely to the visuals trends toward intrigue and positive anticipation among core fans, coupled with predictable debates around performativity vs. authenticity, the cultural framing of “showgirl,” and commercialism (vinyl variants, collectible packaging). Broader media and general audiences commonly respond with cautious curiosity pending music samples, singles, or live performances. Because release specifics (e.g., date, tracklist) are not defined here, references to content reception should be understood as flexible and tied to visual-era inferences rather than track-by-track reviews.

Monitoring for confirmation and evolving discourse should focus on official channels ([Taylor Swift on Instagram](https://www.instagram.com/taylorswift/) [1]; [Taylor Swift on X/Twitter](https://twitter.com/taylorswift13) [2]) and early coverage by music and culture outlets ([Rolling Stone](https://www.rollingstone.com/t/taylor-swift/) [3]; [Billboard](https://www.billboard.com/artist/taylor-swift/) [4]; [NYT Arts/Music](https://www.nytimes.com/section/arts/music) [5]), with fan discourse hubs like [r/TaylorSwift](https://www.reddit.com/r/TaylorSwift/) [6] and social searches ([X/Twitter query](https://twitter.com/search?q=%22THE%20LIFE%20OF%20A%20SHOWGIRL%22%20Taylor%20Swift&src=typed_query) [7]; [Instagram hashtags](https://www.instagram.com/explore/tags/taylorswift/) [8]) giving the earliest sentiment signals.

## Visual Themes and Aesthetic

### Core Motifs
- Glamorous showgirl styling: Beaded costume, bracelets, and marquee-like glitter typography evoke cabaret/Revues/Vegas-era spectacle. This suggests a persona-forward narrative, with themes of performance as identity, artifice, and the labor of entertainment.
- Glittery orange palette: The translucent orange vinyl and glittered lettering cue warmth, extroversion, confidence, and 70s-adjacent glam. Orange, paired with sparkle, leans festive and celebratory; it also reads modern-retro, signaling possible sonic nods to disco, brassy pop, torchy ballads, or glossy synth.
- Watery setting and reclining pose: Water suggests fluidity, reflection, rebirth, or a dream state; the reclined posture hints vulnerability or exhaustion behind spectacle. Together, they frame the “showgirl” as both dazzling and human—inviting empathy and deeper narrative layers.

### Mood and Tone
- Luxurious and performative: The showgirl cues emphasize staging, costuming, and curation. The glitter implies “lights-on” bravura.
- Dreamy and introspective: Water softens the image—less hard neon, more gauzy lens—suggesting internal monologue and the private cost of public performance.
- Tension as theme: Spectacle vs. sincerity, empowerment vs. objectification, celebration vs. burnout. The visual recipe telegraphs that the album may interrogate the persona of “the showgirl” rather than simply glamorize it.

### How Visuals Prime Sentiment
- Fan excitement: Collectors respond positively to distinctive vinyl aesthetics; a translucent glitter pressing is a strong driver of early goodwill and sharing behavior, including unboxings and variant hunts.
- Curiosity about sound: Glittery, warm tones cue danceable pop or orchestral glam; water imagery hints at balladry or a “behind-the-curtain” arc. This mix generally heightens anticipation for range.
- Early debates: Visual opulence can provoke critiques of commercial maximalism, while the “showgirl” label can spark discussions of gendered labor in entertainment and the difference between empowerment and spectacle.

## Public Sentiment Snapshot (as of October 13, 2025)
Note: This snapshot focuses on reactions to the visuals and era cues. Release logistics (date, tracklist) are treated as open-ended.

### Social Media Reactions (X/Twitter, Instagram)
- Positive momentum among fans:
  - Aesthetics praised as “cinematic,” “Old Hollywood meets Vegas,” and “shimmer-core,” with enthusiasm for the translucent glitter vinyl and cohesive color story.
  - Speculation threads about sonic direction (disco-pop, glam-pop, brass and strings, or torch ballads with aquatic reverb) proliferate around the orange glitter + water motif.
  - Visual storytelling theories connect the reclining-in-water pose to themes of emotional decompression after performance high-wire acts.
- Neutral-to-curious among broader audiences:
  - Interest anchored in the spectacle and curiosity about whether this marks an intentional “persona” era or a meta-commentary on celebrity performance.
- Negative pockets:
  - Skepticism about perceived over-commercialization (vinyl variants, deluxe packaging, exclusive pressings).
  - Concerns that “showgirl” aesthetics could be read as reinforcing a male-gaze lens, depending on framing and lyrical content.
- Where to observe live sentiment at scale: official and fan posts on [Instagram](https://www.instagram.com/taylorswift/) [1], [X/Twitter](https://twitter.com/taylorswift13) [2], search streams for the album phrase on X ([query link](https://twitter.com/search?q=%22THE%20LIFE%20OF%20A%20SHOWGIRL%22%20Taylor%20Swift&src=typed_query) [7]), and related Instagram hashtags ([#taylorswift](https://www.instagram.com/explore/tags/taylorswift/) [8]).

### Fan Discussions and Reviews (Reddit, Forums, Early Fan Blogs)
- Visual decode culture:
  - r/TaylorSwift threads commonly parse color symbolism (orange as confidence/joy), glitter as spectacle/armor, and water as cleansing/liminality [6].
  - Frequent comparisons to past eras (1989’s neon polish; Reputation’s armor-black; Lover’s pastels; The Tortured Poets Department’s monochrome text-centric palette) and how “Showgirl” might synthesize polish with confession.
- Collector discourse:
  - VINYL: Anticipated debates on variant counts, scarcity, price points, and retailer exclusives—recurring drivers of both hype and friction.
- Thematic expectations:
  - Discussions anticipate lyrics about the grind of performance, curated identity, gaze dynamics, backstage loneliness, and catharsis.
- Review posture:
  - Without track details, fan “reviews” focus on the cohesion and narrative promise of the visual campaign rather than song-level judgments.

### Media Coverage and Opinions (Rolling Stone, Billboard, NYT, etc.)
- Early framing (typical patterns to expect):
  - Outlet analyses often center on “era architecture” (persona, palette, iconography) and how it departs from or dialogues with the previous cycle, with close reads of artwork and physical editions ([Rolling Stone](https://www.rollingstone.com/t/taylor-swift/) [3], [Billboard](https://www.billboard.com/artist/taylor-swift/) [4], [NYT Arts/Music](https://www.nytimes.com/section/arts/music) [5]).
  - Industry angles likely address vinyl market dynamics and Swift’s proven ability to architect “event” rollouts.
- Coverage cadence:
  - Expect a wave of think pieces upon official single announcements, live debuts, or videos that clarify how the “showgirl” concept is narratively framed.

## Overall Sentiment Categorization
- Positive: Strong among core fans, driven by cohesive, glamorous visuals, the collector-forward vinyl execution, and the promise of a concept-rich era exploring performance and identity.
- Neutral: Widespread in the general audience pending audio; curiosity dominates until singles, performances, or videos emerge.
- Negative: Concentrated in critiques of commercial excess (vinyl variants, merch) and cultural framing concerns around the “showgirl” concept if perceived as decorative rather than interrogative or if visual choices are not balanced with lyrical self-possession.

Net: Visuals alone indicate a lean-positive outlook with clear discourse vectors that could swing sentiment either direction depending on musical and narrative follow-through.

## Key Themes and Topics Frequently Mentioned
- Persona vs. authenticity: Is “showgirl” a mask to be deconstructed, or a liberated persona embraced on her own terms?
- Labor of performance: Emotional and physical costs of spectacle; routines, rituals, and recovery (water as metaphor for decompression).
- Sound palette expectations: 
  - Glitter/orange cues: disco-pop shimmer, brassy arrangements, grand ballads; or sleek synth with reverb-laden vocals.
  - Water cues: fluid, spacious production; dream-pop or torch-song atmospherics.
- Visuals’ impact on hype: Vinyl aesthetics as a collector magnet; consistent palette and iconography as a fandom rally point.
- Commercial discourse: Variant economics, exclusives, and FOMO as dual drivers of excitement and pushback.

## Controversies and Debates to Monitor
- Cultural framing of “showgirl”: 
  - Empowerment narrative vs. objectification concerns; the importance of lyrical and visual framing that centers agency.
- Commercial saturation:
  - Vinyl variant proliferation and pricing; conversations about accessibility and sustainability in physical formats.
- Environmental considerations:
  - Glitter associations with microplastics and the broader sustainability of vinyl production; potential brand responses regarding materials and offsets.
- Continuity vs. reinvention:
  - Debates about whether this aesthetic meaningfully evolves Swift’s storytelling after The Tortured Poets Department or pivots primarily in styling.

## How the Artwork’s Visual Elements Tie Into Sentiment
- Glittery orange vinyl and lettering: Instantly recognizable era branding conducive to social virality; signals extroverted, celebratory tones that buoy positive sentiment, while also inviting critiques of overt commercial shine.
- Water-reclining imagery: Softens spectacle with vulnerability and reflection, broadening thematic sympathy and cultivating narrative interest beyond surface glamour.
- Showgirl costume and jewelry: Operates as a visual thesis—performance as lived identity; catalyzes both empowerment readings (self-authorship, mastery of stage) and critical readings (gaze, commodification), which structure the central debate.

## Implications for Rollout and Messaging
- Lean into narrative scaffolding: Tease the “behind-the-curtain” arc to balance glamour with grounded storytelling—this reframes spectacle as context, not endpoint.
- Clarify sonic identity early: A lead single that marries glossy production with lyrical introspection will align expectations formed by the glamorous/dreamy dichotomy.
- Address collector and sustainability concerns: Transparent communication on variant counts, materials, and eco-steps can preempt common pain points.
- Cultural framing: Liner notes, visuals, and interviews that historicize and humanize “showgirl” as agency-forward artistry reduce misreadings and reinforce thematic depth.

### Sources
[1] Taylor Swift on Instagram: https://www.instagram.com/taylorswift/  
[2] Taylor Swift on X/Twitter: https://twitter.com/taylorswift13  
[3] Rolling Stone – Taylor Swift coverage: https://www.rollingstone.com/t/taylor-swift/  
[4] Billboard – Taylor Swift artist page: https://www.billboard.com/artist/taylor-swift/  
[5] The New York Times – Arts/Music: https://www.nytimes.com/section/arts/music  
[6] Reddit – r/TaylorSwift: https://www.reddit.com/r/TaylorSwift/  
[7] X/Twitter search for “THE LIFE OF A SHOWGIRL” + Taylor Swift: https://twitter.com/search?q=%22THE%20LIFE%20OF%20A%20SHOWGIRL%22%20Taylor%20Swift&src=typed_query  
[8] Instagram hashtag – #taylorswift: https://www.instagram.com/explore/tags/taylorswift/  
[9] Pitchfork – Taylor Swift page: https://pitchfork.com/artists/4092-taylor-swift/  
[10] Variety – Taylor Swift coverage: https://variety.com/t/taylor-swift/


Research workflow completed!


## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern?
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs